In [11]:
"""
Downloads NYPD Complaint Data Historic (Socrata dataset qgea-i56i),
filtered to complaints from 2010-01-01 through 2013-12-31, and saves it
as a single CSV. Matches the date range START/M-START train on
(2010-2013 training split, forecasting 2014 onward).

Run locally (this environment's sandbox can't reach data.cityofnewyork.us):
    pip install requests pandas
    python3 download_nyc_2010_2013.py

Output: nypd_complaints_2010_2013.csv
"""

import time
import requests
import pandas as pd

BASE_URL = "https://data.cityofnewyork.us/resource/qgea-i56i.json"
START_DATE = "2010-01-01T00:00:00"
END_DATE = "2013-12-31T23:59:59"
PAGE_SIZE = 50000          # Socrata's practical page size
OUT_FILE = "nypd_complaints_2010_2013.csv"

# Only pull the columns START/M-START actually use, to keep the file small:
# date, crime type, hour, location description, lat/long.
SELECT_FIELDS = ",".join([
    "cmplnt_num",
    "cmplnt_fr_dt",
    "cmplnt_fr_tm",
    "ofns_desc",
    "pd_desc",
    "law_cat_cd",
    "boro_nm",
    "prem_typ_desc",
    "latitude",
    "longitude",
])

WHERE = f"cmplnt_fr_dt between '{START_DATE}' and '{END_DATE}'"


def download():
    offset = 0
    frames = []
    while True:
        params = {
            "$select": SELECT_FIELDS,
            "$where": WHERE,
            "$order": "cmplnt_fr_dt",
            "$limit": PAGE_SIZE,
            "$offset": offset,
        }
        resp = requests.get(BASE_URL, params=params, timeout=60)
        resp.raise_for_status()
        rows = resp.json()
        if not rows:
            break
        frames.append(pd.DataFrame(rows))
        print(f"Fetched rows {offset:,} - {offset + len(rows):,}")
        offset += len(rows)
        if len(rows) < PAGE_SIZE:
            break
        time.sleep(0.2)  # be polite to the API

    if not frames:
        print("No data returned - check the date filter / dataset ID.")
        return

    df = pd.concat(frames, ignore_index=True)
    df.to_csv(OUT_FILE, index=False)
    print(f"\nSaved {len(df):,} rows to {OUT_FILE}")


if __name__ == "__main__":
    download()

Fetched rows 0 - 50,000
Fetched rows 50,000 - 100,000
Fetched rows 100,000 - 150,000
Fetched rows 150,000 - 200,000
Fetched rows 200,000 - 250,000
Fetched rows 250,000 - 300,000
Fetched rows 300,000 - 350,000
Fetched rows 350,000 - 400,000
Fetched rows 400,000 - 450,000
Fetched rows 450,000 - 500,000
Fetched rows 500,000 - 550,000
Fetched rows 550,000 - 600,000
Fetched rows 600,000 - 650,000
Fetched rows 650,000 - 700,000
Fetched rows 700,000 - 750,000
Fetched rows 750,000 - 800,000
Fetched rows 800,000 - 850,000
Fetched rows 850,000 - 900,000
Fetched rows 900,000 - 950,000
Fetched rows 950,000 - 1,000,000
Fetched rows 1,000,000 - 1,050,000
Fetched rows 1,050,000 - 1,100,000
Fetched rows 1,100,000 - 1,150,000
Fetched rows 1,150,000 - 1,200,000
Fetched rows 1,200,000 - 1,250,000
Fetched rows 1,250,000 - 1,300,000
Fetched rows 1,300,000 - 1,350,000
Fetched rows 1,350,000 - 1,400,000
Fetched rows 1,400,000 - 1,450,000
Fetched rows 1,450,000 - 1,500,000
Fetched rows 1,500,000 - 1,550,000
F

In [12]:
import pandas as pd

df = pd.read_csv("nypd_complaints_2010_2013.csv", nrows=5)

print("COLUMN NAMES:")
print(df.columns.tolist())

print("\nDTYPES:")
print(df.dtypes)

print("\nFIRST 5 ROWS:")
print(df.head().to_string())

COLUMN NAMES:
['cmplnt_num', 'cmplnt_fr_dt', 'cmplnt_fr_tm', 'ofns_desc', 'pd_desc', 'law_cat_cd', 'boro_nm', 'prem_typ_desc', 'latitude', 'longitude']

DTYPES:
cmplnt_num        object
cmplnt_fr_dt      object
cmplnt_fr_tm      object
ofns_desc         object
pd_desc           object
law_cat_cd        object
boro_nm           object
prem_typ_desc     object
latitude         float64
longitude        float64
dtype: object

FIRST 5 ROWS:
       cmplnt_num             cmplnt_fr_dt cmplnt_fr_tm                        ofns_desc                         pd_desc   law_cat_cd        boro_nm            prem_typ_desc   latitude  longitude
0  69526296H17546  2010-01-01T00:00:00.000     02:50:00  MURDER & NON-NEGL. MANSLAUGHTER                          (null)       FELONY       BROOKLYN                   (null)  40.671360 -73.881811
1        69479989  2010-01-01T00:00:00.000     02:00:00   CRIMINAL MISCHIEF & RELATED OF  MISCHIEF, CRIMINAL 4, OF MOTOR  MISDEMEANOR  STATEN ISLAND                   S

In [14]:
%%writefile preprocess_nyc.py
"""
Turns the raw NYPD complaint-level CSV (from download_nyc_2010_2013.py)
into the hourly, per-crime-type count series that START/M-START train on
(Sec IV-B of both papers), and saves sliding-window tensors ready for
MSTART.forward() in mstart_model.py.

Pipeline (matches the papers exactly):
  1. Parse cmplnt_fr_dt + cmplnt_fr_tm into a single datetime.
  2. Drop invalid/missing dates.
  3. Keep only the top-5 most frequent crime types
     (petit larceny, harassment, burglary, criminal mischief, grand larceny)
     -- matched here via NYPD's `ofns_desc` / `pd_desc` text, since NYPD's
     labels don't exactly match Chicago's crime-code names.
  4. Bucket into hourly counts per crime type -> a (T, 5) matrix.
  5. One-hot-encode hour-of-day and location (premise type) per hour bucket
     (majority premise type in that hour, matching the paper's H_onehot).
  6. Build lookback/horizon sliding windows (K, H) and save as .npz,
     directly loadable by demo_train.py / MSTART.forward().

Run:
    pip install pandas numpy
    python3 preprocess_nyc.py --csv nypd_complaints_2010_2013.csv
"""

import argparse
import numpy as np
import pandas as pd


# NYPD ofns_desc / pd_desc values mapped onto START's 5 target crime types.
# Kept as substring matches (case-insensitive) since NYPD text labels vary
# in punctuation/capitalization across years.
CRIME_TYPE_MAP = {
    # NYPD misspells this "HARRASSMENT" (double R) in ofns_desc, but spells
    # it correctly in pd_desc ("HARASSMENT,SUBD..."); both spellings are
    # matched here since either field can carry the signal.
    "petit larceny": ["petit larceny", "larceny,petit", "larceny, petit"],
    "harassment": ["harassment", "harrassment"],
    "burglary": ["burglary"],
    "criminal mischief": ["criminal mischief", "mischief"],
    "grand larceny": ["grand larceny", "larceny,grand", "larceny, grand"],
}
CRIME_TYPES = list(CRIME_TYPE_MAP.keys())  # fixed column order == m in the papers

# NYPD encodes missing values as the literal string "(null)", not NaN.
NULL_TOKEN = "(null)"


def clean_text_col(col: pd.Series) -> pd.Series:
    """Blank out literal '(null)' strings (in addition to true NaN) before matching/one-hot."""
    return col.fillna("").astype(str).str.strip().replace(NULL_TOKEN, "", regex=False)


def map_crime_type(offense_text: str):
    if not isinstance(offense_text, str) or not offense_text:
        return None
    text = offense_text.lower()
    for canon, keywords in CRIME_TYPE_MAP.items():
        if any(kw in text for kw in keywords):
            return canon
    return None


def load_and_clean(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, low_memory=False)

    # 1-2. Build a single datetime, drop invalid rows.
    dt = pd.to_datetime(
        df["cmplnt_fr_dt"].astype(str).str.slice(0, 10) + " " + df["cmplnt_fr_tm"].astype(str),
        errors="coerce",
    )
    df = df.assign(datetime=dt).dropna(subset=["datetime"])

    # NYPD's raw export has a handful of garbage dates (e.g. year 1015,
    # 2099) that make an hourly date_range blow up to millions of hours.
    # Clamp hard to the intended window before that ever happens.
    before = len(df)
    df = df[(df["datetime"] >= "2010-01-01") & (df["datetime"] <= "2013-12-31 23:59:59")]
    dropped = before - len(df)
    if dropped:
        print(f"  Dropped {dropped:,} rows with out-of-range/garbage dates.")
    print(f"  Date range after clamping: {df['datetime'].min()} -> {df['datetime'].max()}")

    # 3. Map onto the 5 target crime types; drop everything else.
    ofns = clean_text_col(df["ofns_desc"])
    pd_desc = clean_text_col(df["pd_desc"])
    text_col = ofns + " " + pd_desc
    df = df.assign(crime_type=text_col.map(map_crime_type)).dropna(subset=["crime_type"])

    df = df.assign(
        hour_bucket=df["datetime"].dt.floor("h"),
        hour_of_day=df["datetime"].dt.hour,
        premise=clean_text_col(df["prem_typ_desc"]).replace("", "UNKNOWN"),
    )
    return df


def build_hourly_series(df: pd.DataFrame):
    full_range = pd.date_range(df["hour_bucket"].min(), df["hour_bucket"].max(), freq="h")

    counts = (
        df.groupby(["hour_bucket", "crime_type"]).size().unstack(fill_value=0).reindex(columns=CRIME_TYPES, fill_value=0)
    )
    counts = counts.reindex(full_range, fill_value=0)  # fill hours with zero complaints

    hour_of_day = counts.index.hour.values.astype(np.int64)

    # majority premise type per hour bucket, coded as a fixed-size integer id
    top_premises = df["premise"].value_counts().index[:31].tolist()  # cap at 31 + "OTHER" -> 32 bins (n_location_bins)
    premise_to_id = {p: i for i, p in enumerate(top_premises)}
    OTHER_ID = len(top_premises)  # index 31 == "OTHER"

    premise_mode = df.groupby("hour_bucket")["premise"].agg(lambda s: s.mode().iat[0] if not s.mode().empty else "OTHER")
    premise_mode = premise_mode.reindex(full_range, fill_value="OTHER")
    loc_ids = premise_mode.map(lambda p: premise_to_id.get(p, OTHER_ID)).values.astype(np.int64)

    return counts[CRIME_TYPES].values.astype(np.float32), hour_of_day, loc_ids, full_range


def stl_decompose_full_series(data: np.ndarray, period: int = 24):
    """
    Runs STL ONCE per crime type over the entire hourly series (matches
    Sec III-B/III-F: "Each crime time series is then decomposed"), instead
    of once per sliding window. This is the single biggest speed lever --
    doing it per-window-per-batch-per-epoch (thousands of redundant STL
    fits) is what makes naive training painfully slow.

    data: (T, m) -> seasonal, trend, resid, each (T, m)
    """
    from statsmodels.tsa.seasonal import STL

    T, m = data.shape
    seasonal = np.zeros_like(data)
    trend = np.zeros_like(data)
    resid = np.zeros_like(data)
    for c in range(m):
        print(f"  STL decomposing crime type {c+1}/{m} ...")
        res = STL(data[:, c], period=period, robust=True).fit()
        seasonal[:, c] = res.seasonal
        trend[:, c] = res.trend
        resid[:, c] = res.resid
    return seasonal, trend, resid


def build_windows(data, hours, locs, seasonal, trend, resid, K, H, stride=1):
    T = len(data)
    X, Hx, Lx, Y, Hy, Ly = [], [], [], [], [], []
    Sx, Tx, Rx = [], [], []
    for s in range(0, T - K - H + 1, stride):
        X.append(data[s : s + K])
        Hx.append(hours[s : s + K])
        Lx.append(locs[s : s + K])
        Y.append(data[s + K : s + K + H])
        Hy.append(hours[s + K : s + K + H])
        Ly.append(locs[s + K : s + K + H])
        Sx.append(seasonal[s : s + K])
        Tx.append(trend[s : s + K])
        Rx.append(resid[s : s + K])
    return (
        np.stack(X), np.stack(Hx), np.stack(Lx),
        np.stack(Y), np.stack(Hy), np.stack(Ly),
        np.stack(Sx), np.stack(Tx), np.stack(Rx),
    )


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="nypd_complaints_2010_2013.csv")
    ap.add_argument("--K", type=int, default=48, help="lookback window (hours), paper default sweeps {48,96,192,336,720}")
    ap.add_argument("--H", type=int, default=12, help="forecast horizon (hours)")
    ap.add_argument("--stride", type=int, default=1)
    ap.add_argument("--out", default="nyc_2010_2013_hourly.npz")
    args, _unknown = ap.parse_known_args()  # tolerates Jupyter/Colab's injected -f kernel.json arg

    print("Loading + cleaning raw complaints ...")
    df = load_and_clean(args.csv)
    print(f"Kept {len(df):,} rows across the top-5 crime types after filtering.")
    for ct in CRIME_TYPES:
        print(f"  {ct}: {(df['crime_type'] == ct).sum():,}")

    print("Building hourly per-crime-type series ...")
    data, hours, locs, full_range = build_hourly_series(df)
    print(f"Hourly series shape: {data.shape}  ({full_range[0]} -> {full_range[-1]})")

    print("Running STL decomposition ONCE on the full series per crime type ...")
    seasonal, trend, resid = stl_decompose_full_series(data, period=24)

    print(f"Building sliding windows (K={args.K}, H={args.H}) ...")
    X, Hx, Lx, Y, Hy, Ly, Sx, Tx, Rx = build_windows(
        data, hours, locs, seasonal, trend, resid, args.K, args.H, args.stride
    )
    print(f"Windows: X={X.shape} Y={Y.shape}")

    np.savez_compressed(
        args.out,
        X=X, Hx=Hx, Lx=Lx, Y=Y, Hy=Hy, Ly=Ly,
        Sx=Sx, Tx=Tx, Rx=Rx,  # precomputed STL: seasonal/trend/resid over the lookback window
        crime_types=np.array(CRIME_TYPES),
    )
    print(f"Saved -> {args.out}")


if __name__ == "__main__":
    main()

Overwriting preprocess_nyc.py


In [18]:
!python3 preprocess_nyc.py --csv nypd_complaints_2010_2013.csv --K 48 --H 12 --out nyc_2010_2013_hourly.npz

Loading + cleaning raw complaints ...
  Date range after clamping: 2010-01-01 00:00:00 -> 2013-12-31 23:59:00
Kept 1,137,766 rows across the top-5 crime types after filtering.
  petit larceny: 331,241
  harassment: 339,660
  burglary: 74,310
  criminal mischief: 191,934
  grand larceny: 200,621
Building hourly per-crime-type series ...
Hourly series shape: (35064, 5)  (2010-01-01 00:00:00 -> 2013-12-31 23:00:00)
Running STL decomposition ONCE on the full series per crime type ...
  STL decomposing crime type 1/5 ...
  STL decomposing crime type 2/5 ...
  STL decomposing crime type 3/5 ...
  STL decomposing crime type 4/5 ...
  STL decomposing crime type 5/5 ...
Building sliding windows (K=48, H=12) ...
Windows: X=(35005, 48, 5) Y=(35005, 12, 5)
Saved -> nyc_2010_2013_hourly.npz


# **m start**

In [19]:
%%writefile mstart_model.py
"""
M-START: START's exact architecture, with every SELF-attention module
(the encoder's Multi-Head Attention, the decoder's masked self-attention,
and the internal self-attention used inside MH-VARA) replaced by a
selective state-space (Mamba) block -- now backed by the real, official
`mamba-ssm` library (Gu & Dao, 2023) instead of a hand-rolled scan.

Design rule used throughout this file (stated explicitly so it's easy to
defend in a report/thesis):

    * Self-attention (Q, K, V all derived from the SAME sequence)
      -> replaced by SelectiveSSM (Mamba).
    * Cross-attention (the decoder's encoder-decoder attention, where Q
      comes from the decoder and K/V come from the encoder)
      -> left as real attention, because it is not self-attention and
      the M-START paper only targets the quadratic self-attention
      bottleneck.

Everything else -- STL decomposition, one-hot crime-type/spatial
encoding, positional encoding, Frequency Attention, the VAR dependency
module, trend damping, level stacking, seasonal/trend/level (S+T+L)
composition, and the encoder/decoder stacking depth -- mirrors START's
Section III (Butt et al., 2025) layer-for-layer.

Reference mapping (paper -> code):
    Sec III-A  Problem formulation           -> MStartConfig
    Sec III-B  ST-Embedding                  -> STEmbedding
    Sec III-C1 Multi-Head Attention (MHA)     -> SelectiveSSM  (REPLACED, mamba_ssm)
    Sec III-C2 Frequency Attention            -> FrequencyAttention
    Sec III-C3 Multi-Head VAR Attention       -> MultiHeadVARAttention
                                                  (self-attn part REPLACED,
                                                   VAR residual kept)
    Sec III-C4 Feed Forward Network           -> FeedForward
    Sec III-D  ST-Decoder (self-attn,         -> DecoderLayer
               encoder-decoder attn,             (self-attn REPLACED,
               trend damping, level stack)        cross-attn kept)
    Sec III-F  STL / Frequency decomposition  -> stl_decompose

Environment note (Kaggle, torch 2.10+cu128, T4 x2, July 2026):
mamba_ssm==2.2.4 was written against an older `transformers` release and
imports two names (`GreedySearchDecoderOnlyOutput`, `SampleDecoderOnlyOutput`)
that were renamed/removed in transformers 5.0. We don't use mamba_ssm's
LM-head/generation utilities at all -- only the `Mamba` block -- so the
patch below just satisfies the import chain without affecting any code
path we actually exercise. This MUST run before `from mamba_ssm import
Mamba`, and must be at module import time (not inside a function), since
mamba_ssm's own __init__.py triggers the offending import as a side
effect of `import mamba_ssm`.
"""

import math
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from statsmodels.tsa.seasonal import STL
    _HAS_STATSMODELS = True
except Exception:  # pragma: no cover
    _HAS_STATSMODELS = False


# --------------------------------------------------------------------------
# Compatibility patch: must run before importing mamba_ssm (see module
# docstring above for why). Safe no-op if the names already exist.
# --------------------------------------------------------------------------
import transformers.generation as _tgen

if not hasattr(_tgen, "GreedySearchDecoderOnlyOutput"):
    _tgen.GreedySearchDecoderOnlyOutput = getattr(_tgen, "GenerateDecoderOnlyOutput", object)
if not hasattr(_tgen, "SampleDecoderOnlyOutput"):
    _tgen.SampleDecoderOnlyOutput = getattr(_tgen, "GenerateDecoderOnlyOutput", object)

from mamba_ssm import Mamba as _MambaCUDA


# --------------------------------------------------------------------------
# Config  (values default to START's Table II / M-START's Table V)
# --------------------------------------------------------------------------
@dataclass
class MStartConfig:
    num_crime_types: int = 5          # m : petit larceny, harassment, burglary, criminal mischief, grand larceny
    d_model: int = 512                # dimensionality of the architecture
    d_ff: int = 2048                  # hidden layer size (FFN)
    encoder_layers: int = 3           # spatiotemporal encoder layers
    decoder_layers: int = 3           # spatiotemporal decoder stacks
    num_var_heads: int = 8            # number of VARA heads
    var_lag_p: int = 4                # VAR lag order
    kernel_size: int = 5              # input kernel size (used in Mamba's causal conv, maps to d_conv)
    lookback_K: int = 96              # input window length
    horizon_H: int = 24               # forecast horizon
    d_state: int = 16                 # Mamba SSM state dimension
    n_hour_bins: int = 24             # one-hot "hour of day"
    n_location_bins: int = 32         # one-hot "location description" bins
    dropout: float = 0.1
    stl_period: int = 24              # dominant seasonal cycle (hourly data -> 24h)


# --------------------------------------------------------------------------
# Sec III-F : STL decomposition  x(t) = s(t) + tau(t) + r(t)
# --------------------------------------------------------------------------
def stl_decompose(x: torch.Tensor, period: int = 24) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Seasonal-Trend decomposition via LOESS (Cleveland et al., 1990), applied
    independently per (batch, crime-type) series, exactly as START/M-START
    describe in eq. (2)/(15).

    x: (B, T, m) -> returns seasonal, trend, resid, each (B, T, m)
    Falls back to a simple moving-average decomposition if statsmodels is
    unavailable or a series is too short for STL.
    """
    B, T, m = x.shape
    device, dtype = x.device, x.dtype
    x_np = x.detach().cpu().numpy()

    seasonal = torch.zeros_like(x)
    trend = torch.zeros_like(x)
    resid = torch.zeros_like(x)

    for b in range(B):
        for c in range(m):
            series = x_np[b, :, c]
            if _HAS_STATSMODELS and T >= 2 * period:
                try:
                    res = STL(series, period=period, robust=True).fit()
                    s, t_, r = res.seasonal, res.trend, res.resid
                except Exception:
                    s, t_, r = _moving_average_decompose(series, period)
            else:
                s, t_, r = _moving_average_decompose(series, period)
            seasonal[b, :, c] = torch.tensor(s, device=device, dtype=dtype)
            trend[b, :, c] = torch.tensor(t_, device=device, dtype=dtype)
            resid[b, :, c] = torch.tensor(r, device=device, dtype=dtype)

    return seasonal, trend, resid


def _moving_average_decompose(series, period):
    import numpy as np
    n = len(series)
    w = max(2, min(period, n - 1))
    kernel = np.ones(w) / w
    trend = np.convolve(series, kernel, mode="same")
    detrended = series - trend
    if n >= period:
        seasonal_pattern = np.array([
            detrended[i::period].mean() if len(detrended[i::period]) > 0 else 0.0
            for i in range(period)
        ])
        seasonal = np.tile(seasonal_pattern, n // period + 1)[:n]
    else:
        seasonal = np.zeros(n)
    resid = series - trend - seasonal
    return seasonal, trend, resid


def stl_level(trend: torch.Tensor) -> torch.Tensor:
    """level = mean of the trend component, eq. (3)/(15)."""
    return trend.mean(dim=1, keepdim=True)


# --------------------------------------------------------------------------
# Sec III-B : ST-Embedding  (one-hot crime-type/spatial encoding + sinusoidal PE)
# --------------------------------------------------------------------------
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding, eq. (2) in START."""

    def __init__(self, d_model: int, max_len: int = 4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class STEmbedding(nn.Module):
    """
    Builds H_onehot from hour-of-day / location-description one-hot vectors
    (eq. 1), embeds the residual crime series, adds positional encoding.
    """

    def __init__(self, cfg: MStartConfig):
        super().__init__()
        self.cfg = cfg
        self.value_proj = nn.Linear(cfg.num_crime_types, cfg.d_model)
        self.hour_embed = nn.Linear(cfg.n_hour_bins, cfg.d_model)
        self.loc_embed = nn.Linear(cfg.n_location_bins, cfg.d_model)
        self.pos_enc = PositionalEncoding(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, residual: torch.Tensor, hour_ids: torch.Tensor, loc_ids: torch.Tensor) -> torch.Tensor:
        """
        residual: (B, T, m)   de-seasonalized/detrended crime series
        hour_ids: (B, T) long in [0, n_hour_bins)
        loc_ids:  (B, T) long in [0, n_location_bins)
        """
        hour_oh = F.one_hot(hour_ids, num_classes=self.cfg.n_hour_bins).float()
        loc_oh = F.one_hot(loc_ids, num_classes=self.cfg.n_location_bins).float()
        h_onehot = self.hour_embed(hour_oh) + self.loc_embed(loc_oh)  # concatenation done additively in-model
        z = self.value_proj(residual) + h_onehot
        z = self.pos_enc(z)
        return self.dropout(z)


# --------------------------------------------------------------------------
# Sec III-C1 (REPLACED): Selective State-Space block == real mamba_ssm.Mamba
# This is the drop-in replacement for START's Multi-Head Self-Attention.
# --------------------------------------------------------------------------
class SelectiveSSM(nn.Module):
    """
    Thin wrapper around the official `mamba_ssm.Mamba` block, kept as its
    own class so every call site in this file (EncoderLayer, DecoderLayer,
    MultiHeadVARAttention) is unchanged from the earlier hand-rolled
    version -- only what happens inside `forward` is now the real,
    CUDA-kernel-backed selective scan instead of a pure-PyTorch scan.

    `mamba_ssm.Mamba` already contains its own in_proj / causal conv1d /
    x_proj (B, C, dt) / selective scan / D-skip / SiLU gating / out_proj --
    i.e. everything the previous hand-rolled SelectiveSSM implemented
    manually. We only add the dropout on the output, since mamba_ssm's
    block doesn't include one and the rest of this file (EncoderLayer,
    DecoderLayer) expects the mixer to have its own dropout, matching how
    it wrapped the old SelectiveSSM.

    `causal` is accepted for API compatibility with the decoder's call
    site (`masked_self_ssm(y, causal=True)`) but has no effect: Mamba's
    scan is a sequential recurrence and is therefore always causal by
    construction, unlike self-attention which needs an explicit mask to
    become causal.
    """

    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2,
                 kernel_size: int = 5, dropout: float = 0.1):
        super().__init__()
        self.mamba = _MambaCUDA(
            d_model=d_model,
            d_state=d_state,
            d_conv=kernel_size,   # mamba_ssm's name for the causal conv kernel width
            expand=expand,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, causal: bool = False) -> torch.Tensor:
        # x: (B, T, d_model) -> (B, T, d_model)
        return self.dropout(self.mamba(x))


# --------------------------------------------------------------------------
# Sec III-C2 : Frequency Attention (kept exactly as in START)
# --------------------------------------------------------------------------
class FrequencyAttention(nn.Module):
    """
    Decomposes the running representation into seasonal/trend components
    and attends over them to recover periodic structure, eq. (7)/(16).
    """

    def __init__(self, d_model: int):
        super().__init__()
        self.Ws = nn.Linear(d_model, d_model)
        self.Wt = nn.Linear(d_model, d_model)
        self.Va = nn.Linear(d_model, 1)

    def forward(self, mha_out: torch.Tensor, seasonal: torch.Tensor, trend: torch.Tensor) -> torch.Tensor:
        h = torch.tanh(self.Ws(seasonal) + self.Wt(trend))     # (B,T,d)
        scores = self.Va(h)                                     # (B,T,1)
        alpha = torch.softmax(scores, dim=1)                    # attention weights over time
        context = (alpha * seasonal).sum(dim=1, keepdim=True)   # (B,1,d)
        return mha_out + context                                # Outputfreq = MHA(...) + c_t


# --------------------------------------------------------------------------
# Sec III-C3 : Multi-Head VAR Attention (self-attention part REPLACED by Mamba,
# VAR lag/residual modelling kept as in both papers, eq. 8/17-19)
# --------------------------------------------------------------------------
class MultiHeadVARAttention(nn.Module):
    """
    MH-VARA: captures cross-crime-type temporal dependencies. START computed
    this with scaled-dot-product self-attention feeding a windowed VAR fit;
    since the attention component here is self-attention (Q,K,V from the
    same z), it is replaced by a SelectiveSSM per this file's design rule.
    The VAR lag/residual mechanics (eq. 8) are kept as a differentiable,
    learnable multivariate autoregression so gradients can flow (a
    statsmodels VAR.fit() inside the network would not be differentiable).
    """

    def __init__(self, cfg: MStartConfig):
        super().__init__()
        self.ssm = SelectiveSSM(cfg.d_model, d_state=cfg.d_state, kernel_size=cfg.kernel_size, dropout=cfg.dropout)
        self.p = cfg.var_lag_p
        # learnable lag coefficient matrices A_1..A_p acting in d_model space
        self.var_lags = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False) for _ in range(self.p)])

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        ssm_out = self.ssm(z)  # replaces "MultiHead(Q,K,V)" in eq. (8)

        B, T, d = z.shape
        var_pred = torch.zeros_like(z)
        for lag, layer in enumerate(self.var_lags, start=1):
            shifted = F.pad(z[:, :-lag, :], (0, 0, lag, 0)) if lag < T else torch.zeros_like(z)
            var_pred = var_pred + layer(shifted)
        epsilon_t = z - var_pred        # eq. (8): residual unexplained by linear VAR structure

        return ssm_out + epsilon_t      # OutputVAR = SelectiveSSM(z) + epsilon_t


# --------------------------------------------------------------------------
# Sec III-C4 : Feed Forward Network (kept exactly as in START, eq. 9)
# --------------------------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.net(x))


# --------------------------------------------------------------------------
# ST-Encoder layer (Sec III-C, Fig. 2/3): MHA -> Mamba, everything else kept
# --------------------------------------------------------------------------
class EncoderLayer(nn.Module):
    def __init__(self, cfg: MStartConfig):
        super().__init__()
        self.self_ssm = SelectiveSSM(cfg.d_model, d_state=cfg.d_state, kernel_size=cfg.kernel_size, dropout=cfg.dropout)  # replaces MHA
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.freq_attn = FrequencyAttention(cfg.d_model)
        self.var_attn = MultiHeadVARAttention(cfg)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, z: torch.Tensor, seasonal: torch.Tensor, trend: torch.Tensor):
        # out1 = LN(X + Dropout(SelfMixer(X)))     [eq. 6, MHA -> SelectiveSSM]
        mixer_out = self.self_ssm(z)
        out1 = self.norm1(z + self.dropout(mixer_out))

        # Frequency attention adds seasonal/trend context onto the mixer output   [eq. 7]
        freq_out = self.freq_attn(mixer_out, seasonal, trend)

        # Multi-Head VAR Attention: cross-crime-type dependencies             [eq. 8]
        var_out = self.var_attn(out1)

        combined = out1 + freq_out + var_out

        # FFN + residual + norm                                              [eq. 9]
        out = self.norm2(combined + self.ffn(combined))
        return out, freq_out, var_out


# --------------------------------------------------------------------------
# ST-Decoder layer (Sec III-D): masked self-attn -> Mamba (causal scan),
# encoder-decoder attention KEPT (it's cross-attention, not self-attention).
# Trend damping / level stacking / frequency attention kept as-is.
# --------------------------------------------------------------------------
class CrossAttention(nn.Module):
    """Standard multi-head encoder-decoder attention (kept, per this file's design rule)."""

    def __init__(self, d_model: int, n_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, q_in: torch.Tensor, kv_in: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.mha(q_in, kv_in, kv_in, need_weights=False)
        return self.norm(q_in + attn_out)


class DecoderLayer(nn.Module):
    def __init__(self, cfg: MStartConfig):
        super().__init__()
        self.masked_self_ssm = SelectiveSSM(cfg.d_model, d_state=cfg.d_state, kernel_size=cfg.kernel_size, dropout=cfg.dropout)  # replaces masked self-attn; Mamba's scan is already causal
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = CrossAttention(cfg.d_model, n_heads=8, dropout=cfg.dropout)  # kept: NOT self-attention
        self.freq_attn = FrequencyAttention(cfg.d_model)

        self.trend_damping = nn.Linear(cfg.d_model, cfg.d_model)
        self.level_stack = nn.Linear(cfg.d_model, cfg.d_model)

        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, y: torch.Tensor, enc_out: torch.Tensor, seasonal: torch.Tensor, trend: torch.Tensor):
        # attn1 = LN(y + Dropout(MaskedSelfMixer(y)))     [eq. 11, self-attn -> SelectiveSSM]
        self_out = self.masked_self_ssm(y, causal=True)
        attn1 = self.norm1(y + self.dropout(self_out))

        # attn2 = encoder-decoder attention (kept as real attention)          [eq. 12]
        attn2 = self.cross_attn(attn1, enc_out)

        # trend damping, level stacking, frequency attention                 [eq. 13]
        trend_damping_out = self.trend_damping(attn2)
        level_stack_out = self.level_stack(attn2)
        freq_out = self.freq_attn(attn2, seasonal, trend)

        combined = trend_damping_out + level_stack_out + freq_out

        # FFN + residual + norm                                             [eq. 14]
        ffn_out = self.norm2(attn2 + self.dropout(self.ffn(combined)))
        return ffn_out, trend_damping_out, level_stack_out


# --------------------------------------------------------------------------
# Full model: ST-Encoder + ST-Decoder + S/T/L composition -> forecast
# --------------------------------------------------------------------------
class MSTART(nn.Module):
    """
    START's exact backbone (ST-Embedding -> STL -> L-layer ST-Encoder ->
    M-layer ST-Decoder -> S+T+L composition -> Linear forecast), with every
    self-attention module swapped for the real mamba_ssm SelectiveSSM block.
    """

    def __init__(self, cfg: MStartConfig):
        super().__init__()
        self.cfg = cfg
        self.embedding = STEmbedding(cfg)
        self.encoder_layers = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.encoder_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.decoder_layers)])

        self.target_embedding = nn.Linear(cfg.num_crime_types, cfg.d_model)
        self.dec_pos_enc = PositionalEncoding(cfg.d_model)

        # final S+T+L composition -> forecast (eq. 9 in M-START / Fig. 2 output stack)
        self.output_proj = nn.Linear(cfg.d_model, cfg.num_crime_types)

    def forward(
        self,
        x_hist: torch.Tensor,          # (B, K, m) lookback window
        hour_ids: torch.Tensor,        # (B, K) long
        loc_ids: torch.Tensor,         # (B, K) long
        dec_hour_ids: torch.Tensor,    # (B, H) long
        dec_loc_ids: torch.Tensor,     # (B, H) long
        teacher_forcing: Optional[torch.Tensor] = None,  # (B, H, m) ground truth for training
        precomputed_stl: Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = None,  # (seasonal, trend, resid), each (B,K,m)
    ) -> torch.Tensor:
        cfg = self.cfg
        B, K, m = x_hist.shape

        # --- Sec III-F: STL decomposition of the input window ---
        # Recomputing STL with statsmodels inside every forward pass is far
        # too slow for real training (thousands of redundant per-window
        # fits). Pass `precomputed_stl` from preprocess_nyc.py, which runs
        # STL ONCE on the full series, matching how the papers actually do
        # it ("Each crime time series is then decomposed" -- once, up
        # front, then windowed).
        if precomputed_stl is not None:
            seasonal, trend, resid = precomputed_stl
        else:
            seasonal, trend, resid = stl_decompose(x_hist, period=cfg.stl_period)
        level = stl_level(trend)  # (B,1,m)

        # --- Sec III-B: ST-Embedding of the de-seasonalized/detrended residual ---
        z = self.embedding(resid, hour_ids, loc_ids)             # (B, K, d)
        seasonal_emb = self.embedding.value_proj(seasonal)        # reuse projector for seasonal/trend context
        trend_emb = self.embedding.value_proj(trend)

        # --- ST-Encoder ---
        for layer in self.encoder_layers:
            z, _, _ = layer(z, seasonal_emb, trend_emb)
        enc_out = z  # (B, K, d)

        # --- Decoder input: start from last observed step, autoregress H steps ---
        H = cfg.horizon_H
        if teacher_forcing is not None:
            dec_in_vals = torch.cat([x_hist[:, -1:, :], teacher_forcing[:, :-1, :]], dim=1)  # (B,H,m)
        else:
            dec_in_vals = x_hist[:, -1:, :].repeat(1, H, 1)

        dec_hour_oh = F.one_hot(dec_hour_ids, cfg.n_hour_bins).float()
        dec_loc_oh = F.one_hot(dec_loc_ids, cfg.n_location_bins).float()
        y = (
            self.target_embedding(dec_in_vals)
            + self.embedding.hour_embed(dec_hour_oh)
            + self.embedding.loc_embed(dec_loc_oh)
        )
        y = self.dec_pos_enc(y)

        dec_seasonal = seasonal_emb[:, -H:, :] if seasonal_emb.size(1) >= H else seasonal_emb.mean(dim=1, keepdim=True).repeat(1, H, 1)
        dec_trend = trend_emb[:, -H:, :] if trend_emb.size(1) >= H else trend_emb.mean(dim=1, keepdim=True).repeat(1, H, 1)

        for layer in self.decoder_layers:
            y, _, _ = layer(y, enc_out, dec_seasonal, dec_trend)

        # --- Final composition: S + T + L -> Linear -> forecast ---
        forecast = self.output_proj(y) + level  # additive level term, as in Fig. 2/3
        return forecast


# --------------------------------------------------------------------------
# Loss / metrics (Sec IV-B: MAPE, MAE, RMSE)
# --------------------------------------------------------------------------
def mape(pred, target, eps=1e-6):
    return torch.mean(torch.abs(target - pred) / (torch.abs(target) + eps))


def mae(pred, target):
    return torch.mean(torch.abs(target - pred))


def rmse(pred, target):
    return torch.sqrt(torch.mean((target - pred) ** 2))

Overwriting mstart_model.py


# **code**

In [22]:
%%writefile train_on_kaggle.py
"""
Trains MSTART on the preprocessed NYC 2010-2013 hourly data
(nyc_2010_2013_hourly.npz, produced by preprocess_nyc.py).

Uses the precomputed STL decomposition (Sx/Tx/Rx) saved in the .npz, so
statsmodels never runs during training -- only mstart_model's Mamba/
attention/VAR layers are on the hot path.

Split is chronological (not random), matching the papers: the earliest
~85% of windows are training, the most recent ~15% are validation, since
shuffling a time series across the split boundary leaks future
information into training.

Mixed precision: fp16 autocast + GradScaler, not bf16. On Turing (T4,
compute capability 7.5) tensor cores get their real throughput benefit
from fp16, and bf16 support on Turing is either absent or software-
emulated depending on op -- unlike Ampere+ where bf16 is the safer
default. fp16 needs the GradScaler (unlike bf16) because its narrower
exponent range can underflow small gradients during backward; bf16 has
the same exponent range as fp32 and doesn't need one. This is the
correct pairing specifically for T4 -- revisit if you ever move to an
Ampere/Hopper GPU (A100/L4/H100), where bf16 + no scaler is preferable.

Run on Kaggle (GPU on):
    !python3 train_on_kaggle.py --npz nyc_2010_2013_hourly.npz --epochs 10 --batch_size 64
"""

import argparse
import time

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from mstart_model import MSTART, MStartConfig, mape, mae, rmse


def load_npz(path):
    d = np.load(path)
    tensors = {
        "X": torch.tensor(d["X"], dtype=torch.float32),
        "Hx": torch.tensor(d["Hx"], dtype=torch.long),
        "Lx": torch.tensor(d["Lx"], dtype=torch.long),
        "Y": torch.tensor(d["Y"], dtype=torch.float32),
        "Hy": torch.tensor(d["Hy"], dtype=torch.long),
        "Ly": torch.tensor(d["Ly"], dtype=torch.long),
        "Sx": torch.tensor(d["Sx"], dtype=torch.float32),
        "Tx": torch.tensor(d["Tx"], dtype=torch.float32),
        "Rx": torch.tensor(d["Rx"], dtype=torch.float32),
    }
    crime_types = d["crime_types"].tolist()
    return tensors, crime_types


def chronological_split(tensors, val_frac=0.15):
    n = tensors["X"].shape[0]
    split = int(n * (1 - val_frac))
    train = {k: v[:split] for k, v in tensors.items()}
    val = {k: v[split:] for k, v in tensors.items()}
    return train, val


def make_loader(t, batch_size, shuffle):
    ds = TensorDataset(t["X"], t["Hx"], t["Lx"], t["Y"], t["Hy"], t["Ly"], t["Sx"], t["Tx"], t["Rx"])
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)


def run_epoch(model, loader, device, opt=None, scaler=None, use_amp=True):
    training = opt is not None
    model.train(training)

    tot_loss = tot_mape = tot_mae = tot_rmse = 0.0
    n_batches = 0

    for X, Hx, Lx, Y, Hy, Ly, Sx, Tx, Rx in loader:
        X, Hx, Lx, Y, Hy, Ly = [t.to(device) for t in (X, Hx, Lx, Y, Hy, Ly)]
        Sx, Tx, Rx = Sx.to(device), Tx.to(device), Rx.to(device)

        with torch.set_grad_enabled(training):
            # Mixed precision forward pass. Metrics (mape/mae/rmse below)
            # are computed outside the autocast context, on the fp32
            # `pred` that autocast automatically casts back on exit, so
            # the reported numbers aren't affected by fp16 rounding in
            # the matmuls themselves.
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                pred = model(
                    X, Hx, Lx, Hy, Ly,
                    teacher_forcing=Y if training else None,
                    precomputed_stl=(Sx, Tx, Rx),
                )
                loss = mae(pred, Y)

        if training:
            opt.zero_grad()
            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)  # unscale before clipping, or the clip threshold is meaningless
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                opt.step()

        tot_loss += loss.item()
        tot_mape += mape(pred, Y).item()
        tot_mae += mae(pred, Y).item()
        tot_rmse += rmse(pred, Y).item()
        n_batches += 1

    return tot_loss / n_batches, tot_mape / n_batches, tot_mae / n_batches, tot_rmse / n_batches


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--npz", default="nyc_2010_2013_hourly.npz")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--d_model", type=int, default=128, help="paper default is 512; 128 is a faster middle ground for Kaggle GPUs")
    ap.add_argument("--d_ff", type=int, default=256)
    ap.add_argument("--encoder_layers", type=int, default=3)
    ap.add_argument("--decoder_layers", type=int, default=3)
    ap.add_argument("--val_frac", type=float, default=0.15)
    ap.add_argument("--save", default="mstart_nyc.pt")
    ap.add_argument("--no_amp", action="store_true", help="disable fp16 mixed precision, run full fp32 (for debugging NaNs)")
    args, _unknown = ap.parse_known_args()  # tolerates Jupyter/Colab's injected -f kernel.json arg

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    use_amp = (device.type == "cuda") and not args.no_amp
    print(f"Mixed precision (fp16 autocast + GradScaler): {use_amp}")

    tensors, crime_types = load_npz(args.npz)
    K = tensors["X"].shape[1]
    H = tensors["Y"].shape[1]
    m = tensors["X"].shape[2]
    print(f"Loaded {tensors['X'].shape[0]:,} windows | K={K} H={H} crime_types={crime_types}")

    train_t, val_t = chronological_split(tensors, val_frac=args.val_frac)
    print(f"Train windows: {train_t['X'].shape[0]:,}  Val windows: {val_t['X'].shape[0]:,}")

    train_loader = make_loader(train_t, args.batch_size, shuffle=True)
    val_loader = make_loader(val_t, args.batch_size, shuffle=False)

    cfg = MStartConfig(
        num_crime_types=m,
        d_model=args.d_model,
        d_ff=args.d_ff,
        encoder_layers=args.encoder_layers,
        decoder_layers=args.decoder_layers,
        lookback_K=K,
        horizon_H=H,
        stl_period=24,
    )
    model = MSTART(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")

    opt = torch.optim.Adam(model.parameters(), lr=args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    scaler = torch.amp.GradScaler(device="cuda", enabled=use_amp)

    best_val_mae = float("inf")
    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        tr_loss, tr_mape, tr_mae, tr_rmse = run_epoch(model, train_loader, device, opt, scaler, use_amp)
        val_loss, val_mape, val_mae, val_rmse = run_epoch(model, val_loader, device, opt=None, scaler=None, use_amp=use_amp)
        sched.step()
        dt = time.time() - t0

        print(
            f"epoch {epoch:2d}/{args.epochs} ({dt:5.1f}s) | "
            f"train MAE={tr_mae:.4f} RMSE={tr_rmse:.4f} MAPE={tr_mape:.4f} | "
            f"val MAE={val_mae:.4f} RMSE={val_rmse:.4f} MAPE={val_mape:.4f}"
        )

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            torch.save({"model_state": model.state_dict(), "cfg": cfg, "crime_types": crime_types}, args.save)
            print(f"  -> new best val MAE, saved to {args.save}")

    print(f"\nDone. Best val MAE: {best_val_mae:.4f}. Model checkpoint: {args.save}")


if __name__ == "__main__":
    main()

Writing train_on_kaggle.py


In [24]:
!python3 train_on_kaggle.py --npz nyc_2010_2013_hourly.npz --epochs 10 --batch_size 64 --no_amp

Device: cuda
Mixed precision (fp16 autocast + GradScaler): False
Loaded 35,005 windows | K=48 H=12 crime_types=['petit larceny', 'harassment', 'burglary', 'criminal mischief', 'grand larceny']
Train windows: 29,754  Val windows: 5,251
Model parameters: 2,152,331
epoch  1/10 ( 25.9s) | train MAE=2.1590 RMSE=3.2453 MAPE=103183.5583 | val MAE=2.4346 RMSE=3.4213 MAPE=102990.8167
  -> new best val MAE, saved to mstart_nyc.pt
epoch  2/10 ( 25.6s) | train MAE=2.0213 RMSE=3.0869 MAPE=94491.3268 | val MAE=2.4173 RMSE=3.4035 MAPE=110024.6421
  -> new best val MAE, saved to mstart_nyc.pt
epoch  3/10 ( 24.9s) | train MAE=1.9440 RMSE=3.0079 MAPE=91326.8906 | val MAE=2.4616 RMSE=3.4430 MAPE=108080.2259
epoch  4/10 ( 25.0s) | train MAE=1.8359 RMSE=2.8889 MAPE=87768.3097 | val MAE=2.5008 RMSE=3.4628 MAPE=107267.9021
epoch  5/10 ( 25.3s) | train MAE=1.7045 RMSE=2.7379 MAPE=82991.4215 | val MAE=2.5742 RMSE=3.5321 MAPE=116849.8670
epoch  6/10 ( 25.1s) | train MAE=1.5654 RMSE=2.5665 MAPE=78023.3850 | val 

START

In [20]:
!cp /kaggle/input/datasets/ishanabeel/start-model/start_model.py /kaggle/working

In [21]:
%%writefile train_start_baseline.py
"""
Trains START (real self-attention, from start_model.py) on the
preprocessed NYC 2010-2013 hourly data (nyc_2010_2013_hourly.npz).

This is an EXACT mirror of the training script that produced your
MSTART numbers -- same loop structure, same fixed 10-epoch run (no
early stopping, no per-step logging), same optimizer/scheduler/split --
so the only variable that differs between the two runs is the model
import (start_model vs mstart_model). This is deliberate: comparing a
fixed-epoch-budget MSTART run against an early-stopped START run would
not be a fair comparison.

Run on Kaggle (GPU on):
    !python3 train_start_baseline.py --npz nyc_2010_2013_hourly.npz --epochs 10 --batch_size 64
"""

import argparse
import time

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from start_model import START, MStartConfig, mape, mae, rmse


def load_npz(path):
    d = np.load(path)
    tensors = {
        "X": torch.tensor(d["X"], dtype=torch.float32),
        "Hx": torch.tensor(d["Hx"], dtype=torch.long),
        "Lx": torch.tensor(d["Lx"], dtype=torch.long),
        "Y": torch.tensor(d["Y"], dtype=torch.float32),
        "Hy": torch.tensor(d["Hy"], dtype=torch.long),
        "Ly": torch.tensor(d["Ly"], dtype=torch.long),
        "Sx": torch.tensor(d["Sx"], dtype=torch.float32),
        "Tx": torch.tensor(d["Tx"], dtype=torch.float32),
        "Rx": torch.tensor(d["Rx"], dtype=torch.float32),
    }
    crime_types = d["crime_types"].tolist()
    return tensors, crime_types


def chronological_split(tensors, val_frac=0.15):
    n = tensors["X"].shape[0]
    split = int(n * (1 - val_frac))
    train = {k: v[:split] for k, v in tensors.items()}
    val = {k: v[split:] for k, v in tensors.items()}
    return train, val


def make_loader(t, batch_size, shuffle):
    ds = TensorDataset(t["X"], t["Hx"], t["Lx"], t["Y"], t["Hy"], t["Ly"], t["Sx"], t["Tx"], t["Rx"])
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)


def run_epoch(model, loader, device, opt=None):
    training = opt is not None
    model.train(training)

    tot_loss = tot_mape = tot_mae = tot_rmse = 0.0
    n_batches = 0

    for X, Hx, Lx, Y, Hy, Ly, Sx, Tx, Rx in loader:
        X, Hx, Lx, Y, Hy, Ly = [t.to(device) for t in (X, Hx, Lx, Y, Hy, Ly)]
        Sx, Tx, Rx = Sx.to(device), Tx.to(device), Rx.to(device)

        with torch.set_grad_enabled(training):
            pred = model(
                X, Hx, Lx, Hy, Ly,
                teacher_forcing=Y if training else None,
                precomputed_stl=(Sx, Tx, Rx),
            )
            loss = mae(pred, Y)

        if training:
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()

        tot_loss += loss.item()
        tot_mape += mape(pred, Y).item()
        tot_mae += mae(pred, Y).item()
        tot_rmse += rmse(pred, Y).item()
        n_batches += 1

    return tot_loss / n_batches, tot_mape / n_batches, tot_mae / n_batches, tot_rmse / n_batches


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--npz", default="nyc_2010_2013_hourly.npz")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--d_model", type=int, default=128, help="paper default is 512; 128 is a faster middle ground for Kaggle GPUs")
    ap.add_argument("--d_ff", type=int, default=256)
    ap.add_argument("--encoder_layers", type=int, default=3)
    ap.add_argument("--decoder_layers", type=int, default=3)
    ap.add_argument("--val_frac", type=float, default=0.15)
    ap.add_argument("--save", default="start_nyc.pt")
    args, _unknown = ap.parse_known_args()  # tolerates Jupyter/Colab's injected -f kernel.json arg

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    tensors, crime_types = load_npz(args.npz)
    K = tensors["X"].shape[1]
    H = tensors["Y"].shape[1]
    m = tensors["X"].shape[2]
    print(f"Loaded {tensors['X'].shape[0]:,} windows | K={K} H={H} crime_types={crime_types}")

    train_t, val_t = chronological_split(tensors, val_frac=args.val_frac)
    print(f"Train windows: {train_t['X'].shape[0]:,}  Val windows: {val_t['X'].shape[0]:,}")

    train_loader = make_loader(train_t, args.batch_size, shuffle=True)
    val_loader = make_loader(val_t, args.batch_size, shuffle=False)

    cfg = MStartConfig(
        num_crime_types=m,
        d_model=args.d_model,
        d_ff=args.d_ff,
        encoder_layers=args.encoder_layers,
        decoder_layers=args.decoder_layers,
        lookback_K=K,
        horizon_H=H,
        stl_period=24,
    )
    model = START(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")

    opt = torch.optim.Adam(model.parameters(), lr=args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)

    best_val_mae = float("inf")
    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        tr_loss, tr_mape, tr_mae, tr_rmse = run_epoch(model, train_loader, device, opt)
        val_loss, val_mape, val_mae, val_rmse = run_epoch(model, val_loader, device, opt=None)
        sched.step()
        dt = time.time() - t0

        print(
            f"epoch {epoch:2d}/{args.epochs} ({dt:5.1f}s) | "
            f"train MAE={tr_mae:.4f} RMSE={tr_rmse:.4f} MAPE={tr_mape:.4f} | "
            f"val MAE={val_mae:.4f} RMSE={val_rmse:.4f} MAPE={val_mape:.4f}"
        )

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            torch.save({"model_state": model.state_dict(), "cfg": cfg, "crime_types": crime_types}, args.save)
            print(f"  -> new best val MAE, saved to {args.save}")

    print(f"\nDone. Best val MAE: {best_val_mae:.4f}. Model checkpoint: {args.save}")


if __name__ == "__main__":
    main()

Writing train_start_baseline.py


In [22]:
!python3 train_start_baseline.py --npz nyc_2010_2013_hourly.npz --epochs 10 --batch_size 64

Device: cuda
Loaded 35,005 windows | K=48 H=12 crime_types=['petit larceny', 'harassment', 'burglary', 'criminal mischief', 'grand larceny']
Train windows: 29,754  Val windows: 5,251
Model parameters: 1,696,139
epoch  1/10 ( 19.8s) | train MAE=2.2429 RMSE=3.3479 MAPE=107329.2199 | val MAE=2.6358 RMSE=3.6727 MAPE=117912.5312
  -> new best val MAE, saved to start_nyc.pt
epoch  2/10 ( 19.6s) | train MAE=2.1041 RMSE=3.1779 MAPE=98620.5733 | val MAE=2.5887 RMSE=3.6316 MAPE=105699.6594
  -> new best val MAE, saved to start_nyc.pt
epoch  3/10 ( 19.5s) | train MAE=2.0711 RMSE=3.1409 MAPE=96856.9321 | val MAE=2.5617 RMSE=3.6126 MAPE=106142.0113
  -> new best val MAE, saved to start_nyc.pt
epoch  4/10 ( 19.6s) | train MAE=2.0466 RMSE=3.1170 MAPE=95919.6921 | val MAE=2.6822 RMSE=3.7907 MAPE=118585.0946
epoch  5/10 ( 19.5s) | train MAE=2.0229 RMSE=3.0905 MAPE=94758.0114 | val MAE=2.6683 RMSE=3.7915 MAPE=104722.3133
epoch  6/10 ( 19.4s) | train MAE=1.9995 RMSE=3.0631 MAPE=94021.2441 | val MAE=2.621

In [28]:
# Check what you're on first — this determines which prebuilt wheel to grab
!python -c "import torch; print(torch.__version__, torch.version.cuda)"

2.10.0+cu128 12.8


In [29]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

name, compute_cap
Tesla T4, 7.5
Tesla T4, 7.5


In [33]:
!pip install packaging ninja

# Set this to whatever nvidia-smi reported above, e.g. "7.5" for T4
%env TORCH_CUDA_ARCH_LIST=7.5

!MAX_JOBS=4 pip install causal-conv1d --no-build-isolation


env: TORCH_CUDA_ARCH_LIST=7.5


In [3]:
%env TORCH_CUDA_ARCH_LIST=7.5
!MAX_JOBS=2 MAMBA_FORCE_BUILD=TRUE pip install "mamba-ssm @ git+https://github.com/state-spaces/mamba.git@v2.2.4" --no-build-isolation

env: TORCH_CUDA_ARCH_LIST=7.5
  Cloning https://github.com/state-spaces/mamba.git (to revision v2.2.4) to /tmp/pip-install-ltrh1eks/mamba-ssm_f66036e3596246c09b17e7ac6dbdddf4
  Running command git clone --filter=blob:none --quiet https://github.com/state-spaces/mamba.git /tmp/pip-install-ltrh1eks/mamba-ssm_f66036e3596246c09b17e7ac6dbdddf4
  Running command git checkout -q 95d8aba8a8c75aedcaa6143713b11e745e7cd0d9
  Resolved https://github.com/state-spaces/mamba.git to commit 95d8aba8a8c75aedcaa6143713b11e745e7cd0d9
  Running command git submodule update --init --recursive -q
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mamba-ssm: filename=mamba_ssm-2.2.4-cp312-cp312-linux_x86_64.whl size=324457319 sha256=59b3e140a352d7bd12defeadfcdd972edace00efb5512e02dabc2f5df06e3f93
  Stored in directory: /tmp/pip-ephem-wheel-cache-pxmnlo94/wheels/e6/92/31/56de471a8952875d5364f80057049311dc249634ea7cdbd0fb
Successfully built mamba-ssm


In [4]:
import transformers.generation as _tgen
if not hasattr(_tgen, "GreedySearchDecoderOnlyOutput"):
    _tgen.GreedySearchDecoderOnlyOutput = getattr(_tgen, "GenerateDecoderOnlyOutput", object)
if not hasattr(_tgen, "SampleDecoderOnlyOutput"):
    _tgen.SampleDecoderOnlyOutput = getattr(_tgen, "GenerateDecoderOnlyOutput", object)

import torch
from mamba_ssm import Mamba
m = Mamba(d_model=128, d_state=16, d_conv=4, expand=2).to("cuda")
x = torch.randn(2, 48, 128, device="cuda")
print(m(x).shape)
print(m.use_fast_path)

torch.Size([2, 48, 128])
True


In [10]:
!find /tmp -name "*.whl" 2>/dev/null | grep -E "mamba_ssm|causal_conv1d"

In [1]:
%%writefile benchmark_mixer.py
"""
Isolated temporal-mixer benchmark: self-attention (START) vs. real Mamba
(M-START), matching the methodology of START Table VI / Fig. 5 -- measure
wall-clock time and peak memory for JUST the mixer, across a range of
sequence lengths, with everything else held fixed. This is the actual
experiment that tests the O(n^2) vs O(n) claim; the earlier end-to-end
training-loop comparisons were NOT this (they were both dominated by
other fixed overhead at K=48, too short to see the asymptotic separation).

Run on Kaggle (GPU on), after mstart_model.py (real Mamba version) is
already on disk:
    !python3 benchmark_mixer.py --d_model 128 --batch_size 32
"""

import argparse
import time

import torch
import torch.nn as nn

from mstart_model import SelectiveSSM  # real mamba_ssm-backed mixer


class SelfAttentionMixer(nn.Module):
    """Standard multi-head self-attention, the thing SelectiveSSM replaces."""

    def __init__(self, d_model, n_heads=8, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

    def forward(self, x):
        out, _ = self.mha(x, x, x, need_weights=False)
        return out


def bench(module, x, n_warmup=5, n_iters=20):
    device = x.device
    module = module.to(device)

    # Warmup: lets CUDA kernels JIT/autotune before timing, and lets the
    # allocator settle so the first real iteration isn't penalized.
    for _ in range(n_warmup):
        y = module(x)
        loss = y.sum()
        loss.backward()
        module.zero_grad(set_to_none=True)
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats(device)

    t0 = time.time()
    for _ in range(n_iters):
        y = module(x)
        loss = y.sum()
        loss.backward()
        module.zero_grad(set_to_none=True)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = (time.time() - t0) / n_iters

    peak_mem_mb = torch.cuda.max_memory_allocated(device) / 1e6 if device.type == "cuda" else float("nan")
    return elapsed * 1000, peak_mem_mb  # ms/iter, MB


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--d_model", type=int, default=128)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--d_state", type=int, default=16)
    ap.add_argument("--kernel_size", type=int, default=5)
    ap.add_argument("--seq_lens", type=int, nargs="+", default=[48, 96, 192, 336, 720, 1440, 2880])
    args, _unknown = ap.parse_known_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type != "cuda":
        print("WARNING: no GPU detected. This benchmark is only meaningful on GPU "
              "(mamba_ssm's kernel requires CUDA anyway).")

    attn = SelfAttentionMixer(args.d_model)
    mamba = SelectiveSSM(args.d_model, d_state=args.d_state, kernel_size=args.kernel_size)

    print(f"\n{'n':>6} | {'Attn ms/iter':>14} | {'Attn mem MB':>12} | {'Mamba ms/iter':>14} | {'Mamba mem MB':>13} | {'Speedup':>8} | {'Mem reduction':>14}")
    print("-" * 100)

    for T in args.seq_lens:
        x = torch.randn(args.batch_size, T, args.d_model, device=device, requires_grad=True)

        try:
            attn_ms, attn_mem = bench(attn, x)
        except torch.cuda.OutOfMemoryError:
            attn_ms, attn_mem = float("nan"), float("nan")
            torch.cuda.empty_cache()

        try:
            mamba_ms, mamba_mem = bench(mamba, x)
        except torch.cuda.OutOfMemoryError:
            mamba_ms, mamba_mem = float("nan"), float("nan")
            torch.cuda.empty_cache()

        speedup = attn_ms / mamba_ms if mamba_ms == mamba_ms else float("nan")
        mem_reduction = attn_mem / mamba_mem if mamba_mem == mamba_mem and mamba_mem > 0 else float("nan")

        print(f"{T:6d} | {attn_ms:14.2f} | {attn_mem:12.1f} | {mamba_ms:14.2f} | {mamba_mem:13.1f} | {speedup:7.2f}x | {mem_reduction:13.1f}x")

    print("\nDone. Compare this table directly against your paper's Table VI / Fig. 5.")


if __name__ == "__main__":
    main()

Writing benchmark_mixer.py


In [2]:
!python3 benchmark_mixer.py --d_model 128 --batch_size 32

Traceback (most recent call last):
  File "/kaggle/working/benchmark_mixer.py", line 21, in <module>
    from mstart_model import SelectiveSSM  # real mamba_ssm-backed mixer
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'mstart_model'


In [3]:
!ls -la /kaggle/working/

total 20
drwxr-xr-x 3 root root 4096 Jul 18 21:25 .
drwxr-xr-x 5 root root 4096 Jul 18 21:20 ..
-rw-r--r-- 1 root root 4129 Jul 18 21:25 benchmark_mixer.py
drwxr-xr-x 2 root root 4096 Jul 18 21:21 .virtual_documents
